# Exploración inicial de datos

**Autora:** Leydy Osorio Vargas  
**Fecha:** 2026-09-07

## Descripción

Este notebook realiza la exploración estructural inicial del conjunto de datos de enfermedad cardiaca almacenado en la capa RAW. El objetivo es comprender su esquema, identificar las formas utilizadas para representar valores nulos, detectar valores incompatibles con el significado de cada variable y convertir las columnas a tipos de datos correctos y uniformes.

Como resultado se genera un dataset intermedio en formato Parquet que servirá como entrada para el análisis exploratorio de la siguiente etapa. En esta actividad no se imputan valores faltantes, no se eliminan duplicados ni valores atípicos y no se realiza modelamiento.

## 📚 Importar librerías

In [1]:
from pathlib import Path

import pandas as pd
import pyarrow as pa

## 💾 Cargar los datos RAW

In [2]:
# Permite ejecutar el notebook tanto desde la raíz del proyecto como desde su carpeta.
PROJECT_DIR = Path.cwd().resolve()

if not (PROJECT_DIR / "data").is_dir():
    PROJECT_DIR = next(
        parent
        for parent in PROJECT_DIR.parents
        if (parent / "data").is_dir() and (parent / "pyproject.toml").exists()
    )

DATA_DIR = PROJECT_DIR / "data"
RAW_FILE = DATA_DIR / "01_raw" / "corazon.csv"

heart_df = pd.read_csv(RAW_FILE, low_memory=False)

print(f"Archivo cargado: {RAW_FILE.relative_to(PROJECT_DIR)}")
print(f"Dimensiones: {heart_df.shape[0]:,} filas y {heart_df.shape[1]} columnas")

Archivo cargado: data/01_raw/corazon.csv
Dimensiones: 3,030 filas y 14 columnas


## 📊 Descripción general de los datos

In [3]:
heart_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3030 entries, 0 to 3029
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   age         3000 non-null   str    
 1   sex         2969 non-null   str    
 2   chest_pain  2947 non-null   str    
 3   rest_bp     2949 non-null   str    
 4   chol        2945 non-null   str    
 5   fbs         2933 non-null   float64
 6   rest_ecg    2837 non-null   str    
 7   max_hr      2859 non-null   str    
 8   exang       2879 non-null   str    
 9   old_peak    2880 non-null   str    
 10  slope       2879 non-null   str    
 11  ca          2868 non-null   str    
 12  thal        2904 non-null   str    
 13  disease     2924 non-null   str    
dtypes: float64(1), str(13)
memory usage: 506.9 KB


In [4]:
missing_summary = pd.DataFrame(
    {
        "data_type": heart_df.dtypes.astype(str),
        "missing_values": heart_df.isna().sum(),
        "missing_percentage": (heart_df.isna().mean() * 100).round(2),
    }
).sort_values("missing_percentage", ascending=False)

missing_summary

,data_type,missing_values,missing_percentage
rest_ecg,str,193,6.37
max_hr,str,171,5.64
ca,str,162,5.35
exang,str,151,4.98
slope,str,151,4.98
old_peak,str,150,4.95
thal,str,126,4.16
disease,str,106,3.50
fbs,float64,97,3.20
chol,str,85,2.81


El conjunto contiene 3.030 registros y 14 variables relacionadas con características demográficas, síntomas, resultados de exámenes y presencia de enfermedad cardiaca. Todas las columnas presentan datos faltantes, aunque ninguna supera el 7 %. La mayor proporción corresponde a `rest_ecg` (6,37 %) y la menor a `age` (0,99 %).

La mayoría de las columnas se interpretaron inicialmente como texto, incluso varias que por su significado deberían ser numéricas, booleanas o categóricas. Por esta razón se revisará primero la representación de los nulos y después la compatibilidad de cada valor con el tipo esperado.

## 🔍 Identificación y unificación de valores nulos

In [5]:
# Se vuelve a leer el CSV como texto y sin inferencia automática de nulos para
# observar exactamente cómo están representados en el archivo original.
raw_text_df = pd.read_csv(
    RAW_FILE,
    dtype="string",
    keep_default_na=False,
)

trimmed_df = raw_text_df.apply(lambda column: column.str.strip())

possible_null_markers = [
    "",
    "?",
    "NA",
    "N/A",
    "NULL",
    "null",
    "None",
    "none",
    "NaN",
    "nan",
]

null_marker_summary = pd.DataFrame(
    {
        marker if marker else "<empty>": trimmed_df.eq(marker).sum()
        for marker in possible_null_markers
    }
)

null_marker_summary = null_marker_summary.loc[:, null_marker_summary.sum() > 0]

null_marker_summary

,<empty>
age,30
sex,61
chest_pain,83
rest_bp,81
chol,85
fbs,97
rest_ecg,193
max_hr,171
exang,151
old_peak,150


Todos los datos faltantes del archivo RAW están representados mediante campos vacíos. No se identificaron otros marcadores como `?`, `NA`, `N/A`, `NULL` o `None`.

Para garantizar una representación uniforme, primero se eliminan espacios accidentales al inicio o al final de los valores y luego los campos vacíos se reemplazan explícitamente por `pd.NA`. En esta etapa los nulos no se imputan y tampoco se eliminan registros.

In [6]:
heart_df = trimmed_df.replace("", pd.NA)
initial_missing = heart_df.isna().sum()

initial_missing.to_frame("missing_values")

,missing_values
age,30
sex,61
chest_pain,83
rest_bp,81
chol,85
fbs,97
rest_ecg,193
max_hr,171
exang,151
old_peak,150


## 🔧 Validación y corrección de tipos de datos

### Clasificación conceptual de las variables

La clasificación se definió a partir del archivo informativo suministrado con el dataset. Las variables `fbs`, `exang` y `disease` tienen dos estados y se tratarán como booleanas. `slope` y `ca` son códigos numéricos discretos con dominios acotados. Las variables que representan etiquetas clínicas se conservarán como categorías.

Aunque el diccionario describe `rest_ecg` mediante códigos 0, 1 y 2, el archivo entregado contiene sus etiquetas textuales (`normal`, `ST-T wave abnormality` y `left ventricular hypertrophy`). Se conservan esas etiquetas porque expresan directamente el significado de cada categoría.

In [7]:
variable_schema = pd.DataFrame(
    [
        ("age", "numérica discreta", "Int16", "Edad en años"),
        ("sex", "categórica nominal", "category", "Male o Female"),
        ("chest_pain", "categórica nominal", "category", "Tipo de dolor torácico"),
        ("rest_bp", "numérica discreta", "Int16", "Presión arterial en reposo"),
        ("chol", "numérica discreta", "Int16", "Colesterol sérico"),
        ("fbs", "booleana", "boolean", "Glucosa en ayunas > 120 mg/dl"),
        ("rest_ecg", "categórica nominal", "category", "Resultado del ECG en reposo"),
        ("max_hr", "numérica discreta", "Int16", "Frecuencia cardiaca máxima"),
        ("exang", "booleana", "boolean", "Angina inducida por ejercicio"),
        ("old_peak", "numérica continua", "Float64", "Depresión del segmento ST"),
        ("slope", "código ordinal", "Int8", "Pendiente del segmento ST: 1, 2 o 3"),
        ("ca", "numérica discreta acotada", "Int8", "Número de vasos: 0 a 3"),
        ("thal", "categórica nominal", "category", "Resultado del estudio thal"),
        ("disease", "booleana (target)", "boolean", "Presencia de enfermedad cardiaca"),
    ],
    columns=["variable", "nature", "target_dtype", "meaning"],
).set_index("variable")

variable_schema

,nature,target_dtype,meaning
variable,,,
age,numérica discreta,Int16,Edad en años
sex,categórica nominal,category,Male o Female
chest_pain,categórica nominal,category,Tipo de dolor torácico
rest_bp,numérica discreta,Int16,Presión arterial en reposo
chol,numérica discreta,Int16,Colesterol sérico
fbs,booleana,boolean,Glucosa en ayunas > 120 mg/dl
rest_ecg,categórica nominal,category,Resultado del ECG en reposo
max_hr,numérica discreta,Int16,Frecuencia cardiaca máxima
exang,booleana,boolean,Angina inducida por ejercicio


### Detección de valores incompatibles

In [8]:
numeric_columns = ["age", "rest_bp", "chol", "max_hr", "old_peak"]

allowed_values = {
    "sex": ["Male", "Female"],
    "chest_pain": ["typical", "asymptomatic", "nonanginal", "nontypical"],
    "fbs": ["0", "1"],
    "rest_ecg": [
        "normal",
        "ST-T wave abnormality",
        "left ventricular hypertrophy",
    ],
    "exang": ["0", "1"],
    "slope": ["1", "2", "3"],
    "ca": ["0.0", "1.0", "2.0", "3.0"],
    "thal": ["normal", "fixed", "reversable"],
    "disease": ["0", "1"],
}

incompatible_records = []

for column in numeric_columns:
    parsed = pd.to_numeric(heart_df[column], errors="coerce")
    invalid_mask = heart_df[column].notna() & parsed.isna()
    for value, count in heart_df.loc[invalid_mask, column].value_counts().items():
        incompatible_records.append(
            {
                "variable": column,
                "expected": variable_schema.loc[column, "target_dtype"],
                "invalid_value": value,
                "count": count,
            }
        )

for column, valid_categories in allowed_values.items():
    invalid_mask = heart_df[column].notna() & ~heart_df[column].isin(valid_categories)
    for value, count in heart_df.loc[invalid_mask, column].value_counts().items():
        incompatible_records.append(
            {
                "variable": column,
                "expected": variable_schema.loc[column, "target_dtype"],
                "invalid_value": value,
                "count": count,
            }
        )

incompatible_details = pd.DataFrame(incompatible_records)

incompatible_summary = (
    incompatible_details.groupby(["variable", "expected"], sort=False)
    .agg(
        incompatible_values=(
            "invalid_value",
            lambda values: ", ".join(sorted(values.astype(str).unique())),
        ),
        incompatible_count=("count", "sum"),
    )
    .reset_index()
)

incompatible_summary

,variable,expected,incompatible_values,incompatible_count
0,age,Int16,"fggfds, sdg",2
1,rest_bp,Int16,"fsgh, wety",2
2,chol,Int16,"sfdywe, wtey",2
3,max_hr,Int16,adfs,1
4,old_peak,Float64,"afd, asd",2
5,sex,category,"2345, 45, 765",3
6,chest_pain,category,"2345, 2435, 3456",3
7,rest_ecg,category,"3563, 36653, 435647, 5653, 5678",5
8,exang,boolean,"adfs, f",2
9,slope,Int8,afd,1


Se encontraron valores claramente incompatibles con el dominio o el tipo de 13 variables. Son cadenas arbitrarias o códigos inexistentes —por ejemplo, `fggfds`, `wety`, `afd`, `fsg` y números usados como categorías— y cada una aparece una sola vez. `fbs` es la única variable cuyo contenido no presenta errores adicionales a sus valores faltantes.

No existe información que permita reconstruir de manera confiable los valores originales. Por ello, los 33 valores incompatibles se reclasifican como faltantes mediante `pd.NA`; no se inventan ni imputan valores.

### Conversión a tipos uniformes

In [9]:
nominal_categories = {
    "sex": ["Male", "Female"],
    "chest_pain": ["typical", "asymptomatic", "nonanginal", "nontypical"],
    "rest_ecg": [
        "normal",
        "ST-T wave abnormality",
        "left ventricular hypertrophy",
    ],
    "thal": ["normal", "fixed", "reversable"],
}

for column, categories in nominal_categories.items():
    heart_df[column] = (
        heart_df[column]
        .where(heart_df[column].isin(categories), pd.NA)
        .astype(pd.CategoricalDtype(categories=categories))
    )

for column in ["age", "rest_bp", "chol", "max_hr"]:
    heart_df[column] = pd.to_numeric(heart_df[column], errors="coerce").astype("Int16")

heart_df["old_peak"] = pd.to_numeric(heart_df["old_peak"], errors="coerce").astype("Float64")

for column, domain in {"slope": [1, 2, 3], "ca": [0, 1, 2, 3]}.items():
    parsed = pd.to_numeric(heart_df[column], errors="coerce")
    heart_df[column] = parsed.where(parsed.isin(domain), pd.NA).astype("Int8")

for column in ["fbs", "exang", "disease"]:
    parsed = pd.to_numeric(heart_df[column], errors="coerce")
    validated = parsed.where(parsed.isin([0, 1]), pd.NA)
    heart_df[column] = validated.map({0: False, 1: True}).astype("boolean")

In [10]:
final_missing = heart_df.isna().sum()

null_comparison = pd.DataFrame(
    {
        "initial_missing": initial_missing,
        "invalid_reclassified": final_missing - initial_missing,
        "final_missing": final_missing,
        "final_missing_percentage": (heart_df.isna().mean() * 100).round(2),
    }
)

null_comparison

,initial_missing,invalid_reclassified,final_missing,final_missing_percentage
age,30,2,32,1.06
sex,61,3,64,2.11
chest_pain,83,3,86,2.84
rest_bp,81,2,83,2.74
chol,85,2,87,2.87
fbs,97,0,97,3.20
rest_ecg,193,5,198,6.53
max_hr,171,1,172,5.68
exang,151,2,153,5.05
old_peak,150,2,152,5.02


In [11]:
type_summary = variable_schema[["nature", "target_dtype"]].copy()
type_summary["result_dtype"] = heart_df.dtypes.astype(str)
type_summary["uniform_type"] = type_summary["target_dtype"] == type_summary["result_dtype"]

type_summary

,nature,target_dtype,result_dtype,uniform_type
variable,,,,
age,numérica discreta,Int16,Int16,True
sex,categórica nominal,category,category,True
chest_pain,categórica nominal,category,category,True
rest_bp,numérica discreta,Int16,Int16,True
chol,numérica discreta,Int16,Int16,True
fbs,booleana,boolean,boolean,True
rest_ecg,categórica nominal,category,category,True
max_hr,numérica discreta,Int16,Int16,True
exang,booleana,boolean,boolean,True


In [12]:
heart_df.info()

arrow_schema = pa.Table.from_pandas(
    heart_df,
    preserve_index=False,
).schema

arrow_schema

<class 'pandas.DataFrame'>
RangeIndex: 3030 entries, 0 to 3029
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   age         2998 non-null   Int16   
 1   sex         2966 non-null   category
 2   chest_pain  2944 non-null   category
 3   rest_bp     2947 non-null   Int16   
 4   chol        2943 non-null   Int16   
 5   fbs         2933 non-null   boolean 
 6   rest_ecg    2832 non-null   category
 7   max_hr      2858 non-null   Int16   
 8   exang       2877 non-null   boolean 
 9   old_peak    2878 non-null   Float64 
 10  slope       2878 non-null   Int8    
 11  ca          2867 non-null   Int8    
 12  thal        2900 non-null   category
 13  disease     2919 non-null   boolean 
dtypes: Float64(1), Int16(4), Int8(2), boolean(3), category(4)
memory usage: 104.3 KB


age: int16
sex: dictionary<values=large_string, indices=int8, ordered=0>
chest_pain: dictionary<values=large_string, indices=int8, ordered=0>
rest_bp: int16
chol: int16
fbs: bool
rest_ecg: dictionary<values=large_string, indices=int8, ordered=0>
max_hr: int16
exang: bool
old_peak: double
slope: int8
ca: int8
thal: dictionary<values=large_string, indices=int8, ordered=0>
disease: bool
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1764

### Registros exactamente duplicados

In [13]:
exact_duplicate_rows = int(heart_df.duplicated().sum())
unique_rows = int(heart_df.drop_duplicates().shape[0])

pd.Series(
    {
        "total_rows": len(heart_df),
        "exact_duplicate_rows": exact_duplicate_rows,
        "unique_rows": unique_rows,
    },
    name="count",
).to_frame()

,count
total_rows,3030
exact_duplicate_rows,2462
unique_rows,568


Se identificaron 2.462 filas exactamente duplicadas. El dataset no contiene un identificador de paciente, por lo que en esta etapa no es posible confirmar si corresponden a repeticiones erróneas o a pacientes distintos con los mismos resultados. Los registros se conservan para mantener la capa intermedia fiel a los datos RAW; su eliminación y el análisis de su impacto pertenecen a las etapas de EDA y Feature Engineering.

## 💾 Guardar el dataset intermedio

In [14]:
INTERMEDIATE_DIR = DATA_DIR / "02_intermediate"
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = INTERMEDIATE_DIR / "corazon_type_fixed.parquet"

heart_df.to_parquet(
    OUTPUT_FILE,
    index=False,
    engine="pyarrow",
    schema=arrow_schema,
)

# Verificación de lectura para comprobar que el archivo conserva datos y tipos.
reloaded_df = pd.read_parquet(OUTPUT_FILE, engine="pyarrow")

assert reloaded_df.shape == heart_df.shape
assert reloaded_df.columns.tolist() == heart_df.columns.tolist()
assert reloaded_df.equals(heart_df)

print(f"Archivo guardado: {OUTPUT_FILE.relative_to(PROJECT_DIR)}")
print(f"Tamaño: {OUTPUT_FILE.stat().st_size / 1024:.1f} KB")
print("Verificación: el Parquet conserva dimensiones, columnas, valores y tipos.")

Archivo guardado: data/02_intermediate/corazon_type_fixed.parquet
Tamaño: 19.2 KB
Verificación: el Parquet conserva dimensiones, columnas, valores y tipos.


## 📊 Análisis de resultados y conclusiones

- El dataset RAW contiene **3.030 filas y 14 columnas**.
- Los faltantes originales estaban representados únicamente por campos vacíos; se unificaron mediante `pd.NA`.
- Se detectaron **33 valores incompatibles** distribuidos en 13 variables. Como no podían reconstruirse de manera confiable, se reclasificaron como faltantes.
- Las variables quedaron representadas mediante tipos uniformes y anulables: `Int16`, `Int8`, `Float64`, `boolean` y `category`.
- No se eliminaron columnas, porque todas corresponden a variables descritas en el diccionario y ninguna supera el 7 % de datos faltantes.
- Se conservaron los **2.462 registros exactamente duplicados** porque no existe un identificador de paciente que permita determinar su origen. Su tratamiento se decidirá en Feature Engineering.
- El archivo `data/02_intermediate/corazon_type_fixed.parquet` se generó y se leyó nuevamente con éxito, preservando dimensiones, valores y tipos.

El resultado cumple el propósito de la capa intermedia: conservar la información original, corregir únicamente problemas estructurales y dejar un esquema consistente para el EDA.

## 💡 Propuestas e ideas para continuar

- En el EDA se debe caracterizar cada variable, estudiar distribuciones, valores atípicos, duplicados y relaciones con `disease`.
- Se debe verificar si los duplicados provienen de una ampliación artificial del dataset o si representan observaciones legítimas antes de eliminarlos.
- La estrategia de imputación debe definirse posteriormente con base en la naturaleza y distribución de cada variable; no debe aplicarse automáticamente en esta etapa.
- Las categorías y rangos observados podrán convertirse después en reglas formales de validación para los pipelines de producción.
- El nombre `reversable` se conserva porque así está definido en el archivo entregado; cambiarlo a `reversible` sería una estandarización semántica que debe documentarse en una etapa posterior.

## 📖 Referencias

- Archivo informativo suministrado con el proyecto: `data/01_raw/datos_corazon_Info.txt`.
- [Evaluación del curso: Ciencia de Datos en Producción](https://joserzapata.github.io/courses/ciencia-datos-en-produccion/intro/evaluacion/).
- [Ejemplo del profesor: Exploración inicial del proyecto Titanic](https://joserzapata.github.io/post/ciencia-datos-proyecto-python/2-exploration/).
- [Tipos de datos anulables de pandas](https://pandas.pydata.org/docs/user_guide/integer_na.html).
- [Integración de pandas con PyArrow](https://pandas.pydata.org/docs/user_guide/pyarrow.html).